In [1]:
from langchain_community.document_loaders import PyPDFLoader
import faiss
from langchain_community.vectorstores import FAISS
from os import listdir, path, chdir

chdir("..")
chdir("..")

from uuid import uuid4

In [3]:
from pydantic_settings import BaseSettings, SettingsConfigDict
from typing import Optional


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env", env_file_encoding="utf-8", extra="ignore"
    )

    AZURE_OPENAI_ENDPOINT: Optional[str] = None
    AZURE_OPENAI_API_KEY: Optional[str] = None
    OPENAI_API_VERSION: Optional[str] = None
    CHAT_MODEL: Optional[str] = None
    EMBEDDING_MODEL: Optional[str] = None

    TAVILY_API_KEY: Optional[str] = None

    LANGGRAPH_SERVER: Optional[str] = None
    CHECKPOINTER: Optional[str] = None


settings = Settings()


In [13]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore

query_store = LocalFileStore("./assets/embedding_cache/query_store/")
docs_store = LocalFileStore("./assets/embedding_cache/docs_store/")

# Consider LLM selection through Configurable

chat_model = ChatOllama(model=settings.CHAT_MODEL)

underlying_embeddings = OllamaEmbeddings(
    model=settings.EMBEDDING_MODEL
)
embedding_model = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    query_embedding_cache=query_store,
    document_embedding_cache=docs_store,
    namespace="embedding_namespace",
)


In [14]:
chat_model.invoke("Hi")

AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-07-06T14:07:09.252937785Z', 'done': True, 'done_reason': 'stop', 'total_duration': 368884746, 'load_duration': 34847897, 'prompt_eval_count': 26, 'prompt_eval_duration': 2901437, 'eval_count': 8, 'eval_duration': 329671047, 'model_name': 'llama3.2:3b'}, id='run--f6e23d4e-db73-4d01-ac8c-033a104be3b0-0', usage_metadata={'input_tokens': 26, 'output_tokens': 8, 'total_tokens': 34})

In [15]:
embedding_model.embed_query("Hi")[0]

0.016826559

#### Identify dataset files

In [16]:
PDF_DIR = "./assets/pdf"

all_files = listdir(PDF_DIR)
pdf_files = [file for file in all_files if file.endswith(".pdf")]
print(pdf_files)

["Diffusion Models for Video Generation _ Lil'Log.pdf", "LLM Powered Autonomous Agents _ Lil'Log.pdf", "Thinking about High-Quality Human Data _ Lil'Log.pdf", "Adversarial Attacks on LLMs _ Lil'Log.pdf", "Extrinsic Hallucinations in LLMs _ Lil'Log.pdf"]


#### Read pdf content

In [17]:
def load_pdf_files(file_path: str):
    for file_name in pdf_files:
        file_path = path.join(PDF_DIR, file_name)
        loader = PyPDFLoader(file_path=file_path)
        docs = loader.load()
        yield file_name, docs

#### Define index

In [18]:
index = faiss.IndexFlatL2(len(embedding_model.embed_query("Hi")))

#### Add documents to Vector Store

In [22]:
try:
    FAISS.load_local("./assets/faiss_index/base" , embeddings=embedding_model, allow_dangerous_deserialization=True)
except Exception as e:
    print(f"Faiss Index not found")
    documents = []
    
    for file_name, docs in load_pdf_files(file_path=PDF_DIR):
        print(f"Loading documents for {file_name}")
        documents.extend(docs)
        
    vector_store = FAISS.from_documents(documents=documents, embedding=embedding_model)

    vector_store.save_local("./assets/faiss_index/base")

Faiss Index not found
Loading documents for Diffusion Models for Video Generation _ Lil'Log.pdf
Loading documents for LLM Powered Autonomous Agents _ Lil'Log.pdf
Loading documents for Thinking about High-Quality Human Data _ Lil'Log.pdf
Loading documents for Adversarial Attacks on LLMs _ Lil'Log.pdf
Loading documents for Extrinsic Hallucinations in LLMs _ Lil'Log.pdf


In [21]:
vector_store.similarity_search("task planning")

[Document(id='a17ca57f-360d-4c63-aac2-1c404eac759a', metadata={'producer': 'Skia/PDF m137', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36', 'creationdate': '2025-06-18T11:50:38+00:00', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'moddate': '2025-06-18T11:50:38+00:00', 'source': "./assets/pdf/LLM Powered Autonomous Agents _ Lil'Log.pdf", 'total_pages': 23, 'page': 1, 'page_label': '2'}, page_content='Figure 1: Overview of a LLM-powered autonomous agent system.\nComponent One: Planning\nA complicated task usually involves many steps. An agent needs to know what they are and\nplan ahead.\nTask Decomposition\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for\nenhancing model performance on complex tasks. The model is instructed to “think step by\nstep” to utilize more test-time computation to decompose hard tasks into smaller and simpler\nsteps. CoT transforms big tasks